In [ ]:
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from games.lunar_lander import LunarLander
from games.flappy_bird import FlappyBirdEnv as FlappyBird
import matplotlib.pyplot as plt
import gymnasium as gym
import os
from create_agents import create_flappy_agent, create_lunarlander_agent, create_robot_agent
from utils import euclideanDist, CE_regression_labels, KL_regression_labels, softmax, from_tensor

### Action Optimality Overview

This file is a workbook that merges ten near-optimal policies to determine if the action chosen by the deciding agent is optimal, near-optimal, or sub-optimal.

#### Data Visualization
The data is visualized with heatmaps and learning curve plots. Heatmaps show the error between the chosen action and the optimal action according to the optimal policy set through an episode. Learning curve plots show the probability that the chosen action from the human agent exists within the set of near-optimal policies over many episodes. 


In [ ]:
error_dict = {
    "FW": {'error':[], 'label': []},
    "RW": {'error':[], 'label': []},
    "LW": {'error':[], 'label': []},
    "FP": {'error':[], 'label': []},
    "RP": {'error':[], 'label': []},
    "LP": {'error':[], 'label': []},
}

heatmap_dict = {
    "FW": {'error':[], 'label': []},
    "RW": {'error':[], 'label': []},
    "LW": {'error':[], 'label': []},
    "FP": {'error':[], 'label': []},
    "RP": {'error':[], 'label': []},
    "LP": {'error':[], 'label': []},
}

In [ ]:
PID = "023_"
condition = "LW"

df_pickle, df_neural, policy_file = find_files(PID, condition)

demonstration = open(df_pickle, 'rb')
demo_dict = dict(pickle.load(demonstration))

seed = demo_dict[1]["seed"]
episodes = demo_dict["NumberOfDemos"]

## Finding Files from Dataset Repo

In [ ]:
def find_files(participant_id, condition):

    source_folder_1 = "/Users/juliasantaniello/Desktop/fNIRS-2-RL/Experiment/ParticipantData/TaskData/raw"
    source_folder_2 = "/Users/juliasantaniello/Desktop/fNIRS-2-RL/Experiment/ParticipantData/fNIRS/FilteredData"

    # Function to find matching files in a folder
    def find_matching_files_with_paths(folder, pid, condition):
        return [
            os.path.join(folder, file)  # Create full path
            for file in os.listdir(folder)
            if (pid in file) and (condition in file)
        ]

    # Search in both folders
    matching_files_folder1 = find_matching_files_with_paths(source_folder_1, participant_id, condition)
    matching_files_folder2 = find_matching_files_with_paths(source_folder_2, participant_id, condition)

    # Combine results
    all_matching_files = {
        "Folder 1": matching_files_folder1,
        "Folder 2": matching_files_folder2
    }

    # Output results
    for folder, files in all_matching_files.items():
        print(f"\nMatching files in {folder}:")
        if files:
            for file in files:
                if file.endswith('.pickle'):
                    df_pickle = file
                if file.endswith('.csv'):
                    df_neural = file
                print(file)
        else:
            print("No matching files found.")

    if condition[0] == "L":
        policy_path = 'policies/LunarLanderPolicies'
    if condition[0] == "F":
        policy_path = 'policies/FlappyBirdPolicies'
    if condition[0] == "R":
        policy_path = 'policies/RobotPolicies'

    return df_pickle, df_neural, policy_path

### Heatmap Code

Plots heatmap for policy agreement over a single episode for some participant and some condition.

In [ ]:
def heatmap(data):
    plt.figure(figsize=(4, 10))  # Adjusts width and height to make it readable
    plt.imshow(data, cmap='viridis', interpolation='nearest', aspect = 'auto')#, vmin=0.0, vmax=10.0)
    plt.colorbar()  # Adds a color bar to the side
    plt.title("Heatmap Comparing Policy Agreeance Over Episodes")
    print(data.shape[0])
    plt.show()


## Running through Demonstration Files

In [ ]:
from math import dist
import numpy as np
def robot_distance(chosen_actions, optimal_actions):
    s=0
    for i, a in enumerate(chosen_actions):
        o = optimal_actions[i]
        if a is None or o is None:
            continue

        distance = dist(a,o)
        s += distance

    return(s)

def robot_optimality_selection(policy_set, state, chosen_action, chosen_action_values, goal=None):
    whole_set_dist, all_errors = [], []

    for policy_index in range(len(policy_set)):
        agent = policy_set[policy_index]

        optimal_action = agent.choose_action(state, goal, train_mode=False)
        optimal_action_values = None

        error = euclideanDist(chosen_action, optimal_action)
      
        all_errors.append(error)
        
    kl_means = np.mean(all_errors[:][0:9], axis=0)
    return kl_means, all_errors


def game_optimality_selection(policy_set, state, chosen_action, chosen_action_values, goal=None):
    prob_set = []
    all_values = []

    for policy_index in range(len(policy_set)):
        agent = policy_set[policy_index]

        _, optimal_action_values = agent.chooseAction(state, 0.0)
        optimal_action_values = softmax(from_tensor(optimal_action_values))
        all_values.append(optimal_action_values)

        # error = optimal_action_values[chosen_action]
        if condition[1] == "W":
            error = KL_regression_labels(chosen_action_values, optimal_action_values)
        else:
            error = CE_regression_labels(chosen_action_values, optimal_action_values)
        # print(error)
        prob_set.append(error)

    # final_set = np.mean(prob_set, axis=0)
    prob_set = np.sort(np.asarray(prob_set))
    final_set = np.mean(prob_set[:][-4:-1], axis=0)

    # mean_values = final_set[chosen_action]

    return final_set, prob_set

### Policy Sets
Creating policy sets with given trained policies.

In [ ]:
def policy_set():
    if condition[0] == "L":
        env = LunarLander
        policy_set = {
            0: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy100_1"),
            1: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy100"),
            2: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98"),
            3: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_1"),
            4: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_2"),
            5: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy98_3"),
            6: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96"),
            7: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_1"),
            8: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_2"),
            9: create_lunarlander_agent("policies/LunarLanderPolicies/LLPolicy96_3")
        }
    if condition[0] == "R":
        env = None
        policy_set = {
            0: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace1.pth"),
            1: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace2.pth"),
            2: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace3.pth"),
            3: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace4.pth"),
            4: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace5.pth"),
            5: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace6.pth"),
            6: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace7.pth"),
            7: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace8.pth"),
            8: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace9.pth"),
            9: create_robot_agent("policies/RobotPolicies/FetchPickAndPlace10.pth"),
        }
        
    if condition[0] == "F":
        env = FlappyBird
        policy_set = {
            0: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy1"),
            1: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy2"),
            2: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy3"),
            3: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy4"),
            4: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy5"),
            5: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy6"),
            6: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy7"),
            7: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy8"),
            8: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy9"),
            9: create_flappy_agent("policies/FlappyBirdPolicies/FlappyBirdOptimalPolicy10"),

        }
    

    return policy_set, env

In [ ]:
def run_lunarlander():
    SET, ENV = policy_set()
    env = ENV()

    optimality_values, policy_agreement_set, labels = [], [], []

    for episode in range(0, episodes):
        steps = demo_dict[episode]["steps"]
        seed = demo_dict[episode]["seed"]
        state = env.reset(seed=seed)

        kls, kls_means = [], []

        for step in range(1, steps):
            try:
                action_prob = demo_dict[episode]["chosen_action_prob"][step]
            except:
                action_prob = demo_dict[episode]["chosen_actions"][step]
                
            action = demo_dict[episode]["actions"][step]
            state = demo_dict[episode]["states"][step - 1]

            optimality_value, agreement_set = game_optimality_selection(SET, state, action, action_prob)

            kls_means.append(optimality_value)
            kls.append(agreement_set)
            
            next_state, reward, done, win = env.step(action)

            if not done:
                n_state = demo_dict[episode]["states"][step]
                    
            state = n_state

            if done:
                if reward > 100:
                    print("Won", episode)
                    labels.append(0)
                if reward >= 15 and reward < 100:
                    print("Sub-optimal Win", episode)
                    labels.append(1)
                if reward < 15:
                    print("Worst-Case", episode)
                    labels.append(2)


        optimality_values.append(kls_means)
        policy_agreement_set.append(kls)

    return policy_agreement_set, optimality_values, labels

def run_flappybird():
    SET, ENV = policy_set()
    env = ENV()

    optimality_values, policy_agreement_set, labels = [], [], []

    for episode in range(0,10): #range(episodes):
        steps = demo_dict[episode]["steps"]
        seed = demo_dict[episode]["seed"]

        try:
            state, _ = env.reset(seed=seed) #reset
        except:
            state = env.reset(seed=seed) #reset

        kls, kls_means = [], []

        for step in range(0, steps):
            try:
                action_prob = demo_dict[episode]["chosen_action_prob"][step]
            except:
                action_prob = demo_dict[episode]["chosen_actions"][step]
                
            action = demo_dict[episode]["actions"][step]
            state = demo_dict[episode]["states"][step-1]

            if action is None:
                continue

            optimality_value, agreement_set = game_optimality_selection(SET, state, action, action_prob)
            kls_means.append(optimality_value)
            kls.append(agreement_set)
    
            next_state, reward_n, done, _, _ = env.step(action)
            
            if not done:
                n_state = demo_dict[episode]["states"][step]

            state = n_state

            if step == steps-1:
                if steps < 75:
                    print("Short Run", episode, step)
                    labels.append(2)
                elif steps < 400:
                    print("Medium Run", episode, step)
                    labels.append(1)
                else:
                    print("Long Run", episode, step)
                    labels.append(0)


        policy_agreement_set.append(kls)
        optimality_values.append(kls_means)

    return policy_agreement_set, optimality_values, labels

def run_robotFetch():
    SET, _ = policy_set()
    env = gym.make('FetchPickAndPlace-v2', max_episode_steps=650)


    optimality_values, policy_agreement_set, labels = [], [], []

    eps = demo_dict["NumberOfDemos"]

    for i in range(0, eps):
        try:
            steps = demo_dict[i]["steps"]
        except:
            continue
        s = robot_distance(demo_dict[i]["chosen_actions"], demo_dict[i]["optimal_actions"])

        
        steps = demo_dict[i]["steps"]
        seed = demo_dict[i]["seed"]

        state_dict, _ = env.reset(seed=seed) #reset
        state = state_dict["observation"]

        kls, kls_means, desired_goals, achieved_goals = [], [], [], []

        for step in range(0, steps):
            state = state_dict["observation"]
            desired_goal = state_dict["desired_goal"]
            achieved_goal = state_dict["achieved_goal"]

            desired_goals.append(desired_goal)
            achieved_goals.append(achieved_goal)

            action = demo_dict[i]["chosen_actions"][step]
            o_action = demo_dict[i]["optimal_actions"][step]
            state = demo_dict[i]["states"][step-1]

            if action is None:
                continue

            corr_10, corr_mean = robot_optimality_selection(SET, state, action, action, desired_goal)
            kls.append(corr_mean)
            kls_means.append(corr_10)
    
            next_state_dict, reward, term, done, _ = env.step(action)

            next_state = next_state_dict["observation"]
            n_desired_goal = next_state_dict["desired_goal"]
            n_achieved_goal = next_state_dict["achieved_goal"]
            state_dict = next_state_dict

            state = next_state
            if step >= steps - 1:
                if reward >= 0.0:
                    print("Optimal", i, s)
                    labels.append(0)
                elif s < 550:
                    print("Sub-Optimal", i, s)
                    labels.append(1)
                else:
                    print("Worst-Case", i, s)
                    labels.append(2)
                
        policy_agreement_set.append(kls)
        optimality_values.append(kls_means)
        demo_dict[i]["desired_goals"] = desired_goals
        demo_dict[i]["achieved_goals"] = achieved_goals

    return policy_agreement_set, optimality_values, labels

In [ ]:
if condition[0] == "L":
    all_errors, all_error_means, labels = run_lunarlander()

if condition[0] == "R":
    all_errors, all_error_means, labels = run_robotFetch()

if condition[0] == "F":
    all_errors, all_error_means, labels = run_flappybird()

In [ ]:
heatmap_dict[condition]['error'] = all_errors
heatmap_dict[condition]['label'] = labels

error_dict[condition]['error'] = all_error_means
error_dict[condition]['label'] = labels
heatmap_df = pd.DataFrame(heatmap_dict)
heatmap_df

error_df = pd.DataFrame(error_dict)
error_df["FP"]['label']

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(1, 3, figsize=(12, 8))

sns.heatmap(np.sort(np.asarray(heatmap_df[condition]['error'][27]), axis=1), ax=axes[0], cmap='icefire', cbar=False)
axes[0].set_title("Optimal", fontsize=24)
axes[0].set_xlabel('Policy #', fontsize=20)
axes[0].set_ylabel('Steps', fontsize=20)
axes[0].tick_params(axis='x', labelsize=18)
axes[0].tick_params(axis='y', labelsize=13)

sns.heatmap(np.sort(np.asarray(heatmap_df[condition]['error'][29]), axis=1), ax=axes[1], cmap='icefire', cbar=False)
axes[1].set_title("Sub-Optimal", fontsize=24)
axes[1].set_xlabel('Policy #', fontsize=20)
axes[1].set_ylabel('Steps', fontsize=20)
axes[1].tick_params(axis='x', labelsize=18)
axes[1].tick_params(axis='y', labelsize=13)

sns.heatmap(np.sort(np.asarray(heatmap_df[condition]['error'][26]), axis=1), ax=axes[2], cmap='icefire', cbar=True, cbar_ax=fig.add_axes([0.92, 0.3, 0.02, 0.4]))
axes[2].set_title("Worst-Case", fontsize=24)
axes[2].set_xlabel('Policy #', fontsize=20)
axes[2].set_ylabel('Steps', fontsize=20)
axes[2].tick_params(axis='x', labelsize=18)
axes[2].tick_params(axis='y', labelsize=13)

fig.suptitle('Policy Agreement (Flappy Passive)', fontsize=30)

plt.tight_layout(rect=[0, 0, 0.9, 0.95])
plt.show()

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(20, 20), sharey='row')
print(error_df["RP"])
for j, col in enumerate(error_df.columns):
    labels, errors = error_df[col]['label'], error_df[col]['error']
    df_group = pd.DataFrame({'label': labels, 'error': errors})

    if col[0] == "R":
        last = 700
    if col[0] == "L":
        last = 500
    if col[0] == "F":
        last = 200
    
    grouped = df_group.groupby('label')['error'].apply(lambda x: np.mean(np.asarray(x).squeeze(), axis=0))
    # grouped = df_group.groupby('label')['error'].apply(lambda x: np.mean(np.vstack(x).reshape(-1, 10), axis=1))
    
    print(grouped.head())
    
    for i, label in enumerate(grouped.index):
        print(f"Label: {label}, Average Error: {grouped[label]}")
        ax = axes[j%6, i%3]
        # ax = axes[i]  # Get the correct Axes object
        ax.set_title(f"{col} - Label {label}")
        ax.plot(pd.Series(grouped[label]).rolling(window=50).mean())
        ax.set_xlabel('Steps')
        ax.set_ylabel('Average Error')

plt.show()
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(5, 8), sharey='col')


error_data_opt = np.asarray(error_df["LP"]['error'][34])
error_data_sub = np.asarray(error_df["LP"]['error'][3])
error_data_worst = np.asarray(error_df["LP"]['error'][7])


axes[0].plot(error_data_opt)
axes[1].plot(error_data_sub)
axes[2].plot(error_data_worst)
ax.set_title(f"FW - Label {error_df['LP']['label'][i]}")
ax.set_xlabel('Steps')
ax.set_ylabel('Error')

plt.tight_layout()
plt.show()


In [ ]:
demo_df

In [ ]:
import seaborn as sns


# Create a DataFrame from the demo_dict
demo_df = pd.DataFrame(demo_dict)
demo_df

# Calculate the average reward for each episode
avg_reward = [demo_df[i]['rewards'][-1] for i in range(episodes)]

# Calculate the average errors for each episode
avg_errors = [np.mean(np.asarray(error_df[condition]['error'][i])) for i in range(episodes)]

# Calculate the running average
window_size = 5
running_avg_reward = np.convolve(avg_reward, np.ones(window_size)/window_size, mode='valid')
running_avg_errors = np.convolve(avg_errors, np.ones(window_size)/window_size, mode='valid')

# Plot the concatenated episode reward averages and running average
sns.set(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(9, 6))

# Plot running average reward
color = '#758e4f'
ax1.set_xlabel('Episode', fontsize=20)
ax1.set_ylabel('Average Reward', fontsize=20)
ax1.plot(range(window_size-1, len(avg_reward)), running_avg_reward, label='Final Reward', linestyle='-', color=color)
ax1.tick_params(axis='y', labelsize=18)
ax1.tick_params(axis='x', labelsize=18)

# Create a second y-axis for the average error
ax2 = ax1.twinx()
color = '#b4436c'
ax2.set_ylabel('Average Error', fontsize=20)
ax2.plot(range(window_size-1, len(avg_errors)), running_avg_errors, label='Average Error', linestyle='-', color=color)
ax2.tick_params(axis='y', labelsize=18)

# Add a legend
fig.legend(loc='lower center', bbox_to_anchor=(0.28, 0.15), fontsize=18)

# Add a title
plt.title('Reward and Policy Agreement over Episodes', fontsize=24)

# Adjust layout
fig.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(6, 3, figsize=(10, 20))

for j, col in enumerate(heatmap_df.columns):
    labels, errors = heatmap_df[col]['label'], heatmap_df[col]['error']
    df_group = pd.DataFrame({'label': labels, 'error': errors})
    
    grouped = df_group.groupby('label')['error'].apply(lambda x: np.mean(np.asarray(x).squeeze(), axis=0))
    
    for i, label in enumerate(grouped.index):
        ax = axes[j % 6, i % 3]
        sns.heatmap(np.array(grouped[label]).reshape(-1, 1), ax=ax, cmap='viridis', cbar=False)
        ax.set_title(f"{col} - Label {label}")
        ax.set_xlabel('Steps')
        ax.set_ylabel('Average Error')

plt.tight_layout()
plt.show()


In [ ]:
heatmap_df[condition]

In [ ]:
fig, axes = plt.subplots(6, 3, figsize=(20, 20))

for j, col in enumerate(error_df.columns):
    labels, errors = error_df[col]['label'], error_df[col]['error']
    df_group = pd.DataFrame({'label': labels, 'error': errors})

    grouped = df_group.groupby('label')['error'].apply(lambda x: np.mean(np.asarray(x).squeeze(), axis=0))
    
    for i, label in enumerate(grouped.index):
        ax = axes[j % 6, i % 3]
        ax.set_title(f"{col} - Label {label}")
        
        # Get the error data for the current label
        error_data = np.array(grouped[label])
        
        # Plot the histogram of errors
        ax.hist(error_data, bins=20, alpha=0.75)
        ax.set_xlabel('Error')
        ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
ep = 0

plt.plot(all_error_means[ep])
print(np.argmax(all_error_means[ep]))

plt.axvline(x=np.argmax(all_error_means[ep]), ymin=0.0, ymax=1.0, color='r')

plt.show

In [ ]:
err = np.sort(np.array(all_errors[29]))

heatmap(err)